In [0]:
import os
import sys
import uuid

sys.path.insert(0, os.path.abspath(".."))  # lets Python find the 'ecom' package in the src folder

from ecom.audit import Audit
from ecom.bronze import run_bronze

dbutils.widgets.text("catalog", "ecom_dev")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("bundle_target", "dev")
dbutils.widgets.text("tables", "")  # empty = all active tables; or a comma list such as: orders,payments

catalog = dbutils.widgets.get("catalog")
run_id = dbutils.widgets.get("run_id") or str(uuid.uuid4())
bundle_target = dbutils.widgets.get("bundle_target")
only = [t.strip() for t in dbutils.widgets.get("tables").split(",") if t.strip()] or None
triggered_by = spark.sql("SELECT current_user()").first()[0]

print(f"catalog={catalog} run_id={run_id} target={bundle_target} tables={only or 'all'}")

In [0]:
audit = Audit(spark, catalog, run_id, "bronze_ingest", bundle_target, triggered_by)
run_bronze(spark, catalog, run_id, audit, only)

In [0]:
display(spark.sql(f"""
    SELECT table_name, status, rows_read, rows_written, rows_rejected, target_version,
           round(duration_sec, 1) AS seconds, error_message
    FROM {catalog}.control.pipeline_run_audit
    WHERE run_id = '{run_id}' AND layer = 'bronze'
    ORDER BY table_name
"""))

In [0]:
%sql SHOW TABLES IN ecom_dev.bronze;

In [0]:
%sql
SELECT table_name, status, count(*) AS files, sum(row_count) AS rows
FROM ecom_dev.control.file_ingestion_log
GROUP BY 1, 2
ORDER BY 1, 2;